# Disciplina 4 — Pesquisa Operacional
## Problema 2: Minimização de Custo (Designação de Equipe)

> ⚠️ **Regra inegociável**: uso de **Excel Solver é proibido**. Modelagem com **PuLP**.

### Cenário (Barbearia Invictus)

A barbearia precisa cobrir **4 turnos** no fim de semana:
- Sábado Manhã, Sábado Tarde
- Domingo Manhã, Domingo Tarde

Tem 3 barbeiros disponíveis (cada um pode trabalhar **no máximo 2 turnos** por questões trabalhistas):

| Barbeiro | Custo/turno | Adicionais |
|---|---:|---|
| Calebe (júnior) | R$ 180 | +R$ 40 no Domingo Tarde |
| Diego (pleno)   | R$ 220 | +R$ 30 no Sábado Manhã |
| Guilherme (sênior) | R$ 260 | +R$ 50 no Domingo Manhã |

### Pergunta

Como **alocar barbeiros aos turnos** para **minimizar o custo** total da folha, garantindo que cada turno tenha exatamente 1 profissional?

In [1]:
# !pip install pulp --quiet
from pulp import (
    LpBinary, LpMinimize, LpProblem, LpStatus, LpVariable,
    PULP_CBC_CMD, lpSum, value
)
import pandas as pd

## 1. Dados de entrada

In [2]:
BARBEIROS_BASE = {
    'Calebe':    (180.0, 2),
    'Diego':     (220.0, 2),
    'Guilherme': (260.0, 2),
}

TURNOS = ['Sab_Manha', 'Sab_Tarde', 'Dom_Manha', 'Dom_Tarde']

ADICIONAL = {
    ('Calebe', 'Dom_Tarde'): 40.0,
    ('Diego',  'Sab_Manha'): 30.0,
    ('Guilherme', 'Dom_Manha'): 50.0,
}

def custo(barbeiro, turno, base):
    return base + ADICIONAL.get((barbeiro, turno), 0.0)

# Matriz de custos para visualizar antes da otimização
matriz = pd.DataFrame(
    {t: [custo(b, t, BARBEIROS_BASE[b][0]) for b in BARBEIROS_BASE] for t in TURNOS},
    index=BARBEIROS_BASE.keys()
)
matriz

,Sab_Manha,Sab_Tarde,Dom_Manha,Dom_Tarde
Calebe,180.0,180.0,180.0,220.0
Diego,250.0,220.0,220.0,220.0
Guilherme,260.0,260.0,310.0,260.0


## 2. Modelagem matemática

**Variável de decisão (binária):**

$$x_{b,t} = \begin{cases} 1 & \text{se barbeiro } b \text{ é alocado ao turno } t \\ 0 & \text{caso contrário} \end{cases}$$

**Função objetivo (minimizar):**

$$\min Z = \sum_{b} \sum_{t} c_{b,t} \cdot x_{b,t}$$

**Restrições:**

$$\sum_{b} x_{b,t} = 1 \quad \forall t \quad \text{(cada turno coberto por exatamente 1 barbeiro)}$$

$$\sum_{t} x_{b,t} \leq 2 \quad \forall b \quad \text{(limite trabalhista)}$$

In [3]:
def resolver(barbeiros: dict) -> dict:
    model = LpProblem('Min_Custo_Folha', LpMinimize)

    x = {(b, t): LpVariable(f'x_{b}_{t}', cat=LpBinary)
         for b in barbeiros for t in TURNOS}

    model += lpSum(custo(b, t, barbeiros[b][0]) * x[(b, t)]
                  for b in barbeiros for t in TURNOS), 'Custo_Total'

    for t in TURNOS:
        model += lpSum(x[(b, t)] for b in barbeiros) == 1, f'Cobertura_{t}'

    for b, (_, max_turnos) in barbeiros.items():
        model += lpSum(x[(b, t)] for t in TURNOS) <= max_turnos, f'Limite_{b}'

    model.solve(PULP_CBC_CMD(msg=False))

    escala = {t: next((b for b in barbeiros if value(x[(b, t)]) > 0.5), '—')
             for t in TURNOS}
    return {
        'status': LpStatus[model.status],
        'escala': escala,
        'custo_total': float(value(model.objective) or 0.0),
    }

## 3. Cenário Base — 3 barbeiros disponíveis

In [4]:
base = resolver(BARBEIROS_BASE)
print(f"Status do solver: {base['status']}")
print(f"CUSTO TOTAL DA FOLHA: R$ {base['custo_total']:,.2f}\n")
pd.Series(base['escala'], name='barbeiro').to_frame()

Status do solver: Optimal
CUSTO TOTAL DA FOLHA: R$ 800.00



,barbeiro
Sab_Manha,Calebe
Sab_Tarde,Calebe
Dom_Manha,Diego
Dom_Tarde,Diego


## 4. Análise de Cenário (What-If) — *obrigatório pelo guia*

**Pergunta:** *"E se Calebe (o mais barato) ficar indisponível?"*

Os outros barbeiros precisam aumentar o limite de turnos para 3 cada (cobrir o gap).

In [5]:
sem_calebe = {b: (v[0], 3) for b, v in BARBEIROS_BASE.items() if b != 'Calebe'}
crise = resolver(sem_calebe)
print(f"Status: {crise['status']}")
print(f"CUSTO TOTAL DA FOLHA: R$ {crise['custo_total']:,.2f}\n")
pd.Series(crise['escala'], name='barbeiro').to_frame()

Status: Optimal
CUSTO TOTAL DA FOLHA: R$ 920.00



,barbeiro
Sab_Manha,Guilherme
Sab_Tarde,Diego
Dom_Manha,Diego
Dom_Tarde,Diego


In [6]:
delta = crise['custo_total'] - base['custo_total']
delta_pct = (delta / base['custo_total']) * 100
print(f'A folha sobe R$ {delta:,.2f} ({delta_pct:.1f}%) sem o barbeiro mais barato.')

A folha sobe R$ 120.00 (15.0%) sem o barbeiro mais barato.


## 5. Análise Crítica — parte 2/2 (Minimização)

> A pergunta obrigatória do guia (item 4.3) é **conjunta** para os dois
> problemas. A resposta consolidada está em
> [`docs/d4_pesquisa_operacional.md`](../docs/d4_pesquisa_operacional.md) § 4.3.

**Os resultados fazem sentido para a barbearia?** Sim. O solver atribui
o **júnior (Calebe)** aos dois turnos do sábado e o **pleno (Diego)**
aos dois turnos do domingo, totalizando **R$ 800/fim de semana** — o
mínimo possível dado que cada turno custa pelo menos R$ 180 e há 4
turnos a cobrir. O sênior (Guilherme), apesar de competente, fica de
fora porque seu custo é desproporcional ao mix atual.

**Como o What-If ajuda a gerência?** Mostrou que perder o barbeiro mais
barato eleva a folha em **15%** (de R$ 800 para R$ 920). Isso confirma
um risco do plano do TAP: a empresa é **economicamente dependente de
profissionais juniores**. A decisão estratégica derivada é manter sempre
**ao menos um barbeiro júnior fixo** na escala e considerar a
contratação de um **segundo júnior como reserva**, antes que o
crescimento da demanda obrigue a usar o sênior em todos os turnos.